# Data Science Lifecycle: Diabetes Risk Prediction

This notebook is the formal lifecycle record for the project, making the workflow easier to inspect like a real data science analysis: context, code, outputs, and interpretation sit together.

## 1. Problem Definition and Business Understanding

The goal is to predict whether a patient will test positive for diabetes using routine health measurements. This is not a diagnostic tool; it is a screening-style analysis that asks whether simple measurements can help prioritize patients for follow-up testing.

The main evaluation metrics are AUC, recall, and precision. Recall matters because missed diabetes cases are costly. Precision matters because false alarms can lead to unnecessary follow-up.

## 2. Data Collection and Sourcing

The dataset is the Pima Indians Diabetes dataset, originally from the National Institute of Diabetes and Digestive and Kidney Diseases. It contains 768 records, 8 predictor measurements, and one outcome column indicating diabetes status.

Important limitation: the dataset only represents women aged 21 and older from the Pima community, so the results should not be assumed to generalize to other populations.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data_prep import ZERO_AS_MISSING, load_raw, load_clean

raw = load_raw(PROJECT_ROOT / 'data' / 'pima_diabetes_raw.csv')
clean = load_clean(PROJECT_ROOT / 'data' / 'pima_diabetes_raw.csv')

raw.shape, clean.shape

## 3. Data Cleaning and Preparation

Several columns use `0` as a disguised missing value. For example, a living patient cannot have a blood pressure or BMI of zero. The cleaning step converts those values to missing values, then the modeling pipeline uses median imputation.

In [ ]:
missing_zero_counts = (raw[ZERO_AS_MISSING] == 0).sum().sort_values(ascending=False)
missing_zero_percent = (missing_zero_counts / len(raw) * 100).round(1)

missing_summary = missing_zero_counts.to_frame('zero_count')
missing_summary['percent_of_rows'] = missing_zero_percent
missing_summary

The largest issue is `insulin`, followed by `skin_thickness`. Because those two columns have high missingness, the project keeps `insulin_missing` and `skin_thickness_missing` flags as additional model features.

## 4. Exploratory Data Analysis

EDA checks whether each measurement appears related to diabetes before training a model. The clearest signals are glucose, BMI, and age.

In [ ]:
outcome_balance = clean['diabetes'].value_counts().sort_index().rename(index={0: 'No diabetes', 1: 'Diabetes'})
outcome_rate = (outcome_balance / len(clean) * 100).round(1)
outcome_balance.to_frame('count').assign(percent=outcome_rate)

In [ ]:
numeric_cols = ['pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree', 'age']
clean[numeric_cols + ['diabetes']].corr()['diabetes'].drop('diabetes').sort_values(ascending=False).round(3)

## 5. Modeling and Experimentation

Two models are compared: Logistic Regression as an interpretable baseline, and Random Forest as a more flexible tree-based model. The split is stratified so the positive diabetes rate stays similar in training and test data.

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from model import build_logistic_pipeline, build_rf_pipeline, split_data

X_train, X_test, y_train, y_test = split_data(clean)
models = {
    'Logistic Regression': build_logistic_pipeline().fit(X_train, y_train),
    'Random Forest': build_rf_pipeline().fit(X_train, y_train),
}

rows = []
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        'model': name,
        'auc': round(roc_auc_score(y_test, proba), 3),
        'recall': round(recall_score(y_test, pred), 3),
        'precision': round(precision_score(y_test, pred), 3),
    })

pd.DataFrame(rows)

## 6. Insights, Visualization, and Storytelling

Glucose is the dominant predictor, followed by BMI and age. The threshold-tradeoff plot is especially important for a screening project because changing the cutoff can catch more true diabetes cases at the cost of more false alarms.

Key figures are saved in `results/figures/`, including missing-value counts, before/after cleaning histograms, ROC curves, confusion matrices, threshold tradeoff, and feature importance.

## 7. Documentation and Deployment

The project is documented with a README, source scripts, generated outputs, and this lifecycle notebook. It is not deployed as a medical tool. Real deployment would require broader validation, clinical review, threshold selection based on real costs, monitoring, and periodic re-evaluation.